# 🧪 Getting Started with the Open Source Synthetic Data SDK

**Video:** https://www.youtube.com/watch?v=jM-fkYC8_LA

The [**Mostly AI Synthetic Data SDK**](https://github.com/mostly-ai/mostlyai) is a fully
permissive **Apache 2.0** open-source library. In two steps you turn a real, sensitive
tabular dataset into a **privacy-safe synthetic replica** that looks, feels, and *means* the
same as the original — easier to access, share, and consume.

The workflow has three primitives:

| Primitive | What it does |
|---|---|
| `mostly.train(...)` | Trains a generative model **from scratch** on your data, then deletes the real data. Returns a *generator* (model weights + metadata). |
| `mostly.probe(...)` | Draws live, disposable synthetic samples from a generator — no dataset materialized. |
| `mostly.generate(...)` | Creates a persisted synthetic dataset you can write back to Unity Catalog. |

This notebook reproduces the demo from the video end-to-end on the **US Census Income**
dataset (~50k rows, 15 columns).


## 1. Install the SDK

It is a plain Python library — `pip install` it in any Python environment. Use the
`[local]` extra to train and generate on the compute you already have. **Restart the
Python kernel** after installing (the video does the same).


In [ ]:
%pip install -U 'mostlyai[local]'
dbutils.library.restartPython()


## 2. Initialize the SDK in local mode

`local` mode runs training and generation on your own compute (your LTAP / Databricks
cluster). `client` mode instead talks to a Mostly AI platform deployed in your secure
environment.


In [ ]:
from mostlyai.sdk import MostlyAI

mostly = MostlyAI()  # LOCAL mode is the default


## 3. Read the real (sensitive) production data

A tabular dataset — CSV, Parquet, a Spark/Unity Catalog table, a relational DB. Here we
use the US Census Income dataset. The SDK learns the **joint probability distributions**
and then deletes the real data (privacy-first).


In [ ]:
import pandas as pd

# US Census Income (public demo data). Swap for your own UC table, e.g.:
#   df_original = spark.read.table('main.default.customers').toPandas()
df_original = pd.read_csv(
    'https://raw.githubusercontent.com/mostly-ai/public-demo-data/dev/census/census.csv.gz'
)
print(df_original.shape)
df_original.head()


## 4. Train a generator

`mostly.train` pulls, analyzes, and encodes the training data, learns its patterns, and
returns a **generator**. For the demo we cap `max_training_time` to 1 minute; set it to
`None` for maximum accuracy. The training log's validation loss flattens as the model
saturates on the data.


In [ ]:
g = mostly.train(config={
    'name': 'US Census Income',
    'tables': [{
        'name': 'census',
        'data': df_original,
        'tabular_model_configuration': {
            'max_training_time': 1,  # minutes; cap for the demo, use None for best accuracy
        },
    }],
})


## 5. Inspect the model report

The two questions everyone asks about synthetic data are **“is it accurate?”** and
**“is it private?”** The generated HTML report answers both — overall accuracy score,
univariate & bivariate distributions (original in gray vs. synthetic in green),
a correlation matrix, cosine similarity for high-dimensional spaces, and distance-based
privacy metrics.


In [ ]:
g.reports(display=True)


## 6. Probe for live samples

`mostly.probe` draws statistically representative, **fully disposable** samples without
materializing a dataset — probe again and you get fresh rows each time. Great for
inspecting data on the fly.


In [ ]:
df_samples = mostly.probe(g, size=100_000)
print(df_samples.shape)
df_samples.head()


## 7. Conditional generation (“better than real” data)

The beauty of a generator is that you can **mold and manipulate** the output. Feed a
**seed** that fixes some columns as real and synthesize the rest — here we single in on
**35-year-old males**. This is exactly how Swiss Post fixed real geo-coordinates at an
area-code level and synthesized every other customer attribute, going from 11% to 100%
of their customer base in usable (synthetic) form.


In [ ]:
seed = pd.DataFrame({
    'age': [35] * 10_000,
    'sex': ['Male'] * 10_000,
})
df_conditional = mostly.probe(g, seed=seed)
df_conditional[['age', 'sex']].value_counts().head()


## 8. Generate a synthetic dataset and write it back to Unity Catalog

`mostly.generate` produces a persisted synthetic dataset. Read it with `.data()` and
save it as a governed table. Because it carries **no real customer information**, you can
share it freely with BI users, data scientists, and external partners.


In [ ]:
sd = mostly.generate(g, size=10_000)
df_synthetic = sd.data()

# Persist to Unity Catalog (adjust catalog.schema.table to your environment)
(spark.createDataFrame(df_synthetic)
      .write.mode('overwrite')
      .saveAsTable('main.default.census_income_synthetic'))

df_synthetic.head()


## 9. (Optional) Differential privacy

All output is private by default. For an additional mathematical guarantee, add a
`differential_privacy` block to the training configuration. You can also train
**multi-table** and **time-series** generators (configure primary/foreign keys) and
text columns via `language_model_configuration`.


In [ ]:
g_dp = mostly.train(config={
    'name': 'US Census Income (DP)',
    'tables': [{
        'name': 'census',
        'data': df_original,
        'tabular_model_configuration': {
            'max_training_time': 1,
            'differential_privacy': {
                'max_epsilon': 5.0,  # stopping criterion
                'delta': 1e-5,
            },
        },
    }],
})


## Wrap-up

`pip install` → `train` → `probe` / `generate`. That is the whole workflow. Open source,
no fees beyond compute, and a privacy-safe replica you can share without exposing real
customer data.

**Resources**
- GitHub: https://github.com/mostly-ai/mostlyai
- Docs & usage: https://mostly-ai.github.io/mostlyai/usage/
- Databricks Free Edition: https://www.databricks.com/learn/free-edition
